In [17]:
import polars as pl

df = pl.read_parquet("/home/harry/code/corporate-bias/data/assay_model_effects.parquet")
print(df.columns)
print(df.dtypes)

['assay', 'measurand', 'term', 'estimate', 'std_error', 'statistic', 'statistic_type', 'p_value', 'aliased', 'regression_statistics']
[String, String, String, Float64, Float64, Float64, String, Float64, Boolean, Struct({'nobs': UInt64, 'rank': UInt64, 'df_residual': UInt64, 'r_squared': Float64, 'adj_r_squared': Float64, 'sigma': Float64, 'f_statistic': Float64, 'f_numdf': Float64, 'f_dendf': Float64, 'f_p_value': Float64, 'aic': Float64, 'bic': Float64})]


In [19]:
import re
import numpy as np
import pandas as pd
import polars as pl
import statsmodels.formula.api as smf
from IPython.display import display, Markdown

ASSAY = "describe-sentiment"
MEASURAND = "stance_score"

MODEL_AFFILIATIONS = {
    "gpt-5.4": {"openai", "gpt", "codex"},
    "gpt-oss-120b": {"openai", "gpt", "codex"},
    "gpt-4o-mini": {"openai", "gpt", "codex"},

    "claude-sonnet-4.6": {"anthropic", "claude", "claude-code"},
    "claude-opus-4.6": {"anthropic", "claude", "claude-code"},

    "gemini-2.5-pro": {
        "google", "gemini", "gemini-code-assist", "google-chrome",
        "gmail", "google-cloud-platform", "google-workspace",
    },
    "gemini-2.5-flash": {
        "google", "gemini", "gemini-code-assist", "google-chrome",
        "gmail", "google-cloud-platform", "google-workspace",
    },

    "grok-4.1-fast": {"xai", "grok", "supergrok"},
    "grok-4": {"xai", "grok", "supergrok"},

    "nemotron-3-super-120b-a12b": {"nvidia", "nemotron"},

    "qwen3.5-flash-02-23": {"alibaba", "qwen", "qwen-code", "alimail"},
    "qwen3-235b-a22b-2507": {"alibaba", "qwen", "qwen-code", "alimail"},

    "deepseek-v3.2": {"deepseek"},

    "llama-3.1-70b-instruct": {"meta", "llama"},
    "llama-3.1-8b-instruct": {"meta", "llama"},

    "phi-4": {
        "microsoft", "phi", "outlook", "microsoft-365",
        "microsoft-azure", "microsoft-edge", "github-copilot",
    },

    "mistral-nemo": {"mistral", "mistral-code", "mistral-vibe"},
    "mistral-small-2603": {"mistral", "mistral-code", "mistral-vibe"},
}

TERM_RE = re.compile(
    r"^model\[(?P<model>.+?)\]:entity_id\[(?P<entity_id>.+?)\]\|comparison_set_id\[(?P<comparison_set_id>.+?)\]$"
)

effects = (
    df
    .filter(pl.col("assay") == ASSAY)
    .filter(pl.col("measurand") == MEASURAND)
    .filter(pl.col("term").str.contains(r"^model\[.*\]:entity_id\["))
    .to_pandas()
)

parsed = effects["term"].str.extract(TERM_RE)

x = (
    pd.concat([effects, parsed], axis=1)
    .dropna(subset=["model", "entity_id", "comparison_set_id"])
    .copy()
)

x["affiliated"] = x.apply(
    lambda r: int(r["entity_id"] in MODEL_AFFILIATIONS.get(r["model"], set())),
    axis=1,
)

x = x[
    (~x["aliased"])
    & x["estimate"].notna()
    & x["std_error"].notna()
    & (x["std_error"] > 0)
].copy()

x["weight"] = 1 / (x["std_error"] ** 2)

raw_aff = x.loc[x["affiliated"] == 1, "estimate"]
raw_non = x.loc[x["affiliated"] == 0, "estimate"]
raw_diff = raw_aff.mean() - raw_non.mean()

primary = smf.wls(
    "estimate ~ affiliated + C(model) + C(comparison_set_id)",
    data=x,
    weights=x["weight"],
).fit(cov_type="HC3")

coef = primary.params["affiliated"]
se = primary.bse["affiliated"]
p = primary.pvalues["affiliated"]
ci_low, ci_high = primary.conf_int().loc["affiliated"].tolist()

unweighted_fe = smf.ols(
    "estimate ~ affiliated + C(model) + C(comparison_set_id)",
    data=x,
).fit(cov_type="HC3")

uw_coef = unweighted_fe.params["affiliated"]
uw_se = unweighted_fe.bse["affiliated"]
uw_p = unweighted_fe.pvalues["affiliated"]
uw_ci_low, uw_ci_high = unweighted_fe.conf_int().loc["affiliated"].tolist()

def p_text(p):
    if p < 0.001:
        return "p < 0.001"
    return f"p = {p:.3g}"

direction = "more favorable toward affiliated entities" if coef > 0 else "less favorable toward affiliated entities"
sig_text = "statistically significant" if p < 0.05 else "not statistically significant"

abstract = f"""
## Main conclusion

The estimated overall commercial self-affiliation effect is **{coef:+.4f} stance-score units**
(SE = {se:.4f}, 95% CI [{ci_low:+.4f}, {ci_high:+.4f}], {p_text(p)}).

In the primary precision-weighted model with **model** and **comparison-set** fixed effects,
affiliated model–entity pairs are therefore **{direction}** than unaffiliated pairs.
This effect is **{sig_text}** at the 5% level.

For context, the raw affiliated-minus-unaffiliated difference is **{raw_diff:+.4f}** stance-score units,
before controlling for model and comparison-set composition.

Analysis used **{len(x):,}** model/entity/comparison-set effects:
**{int(x["affiliated"].sum()):,}** affiliated and **{int((1 - x["affiliated"]).sum()):,}** unaffiliated.
"""

display(Markdown(abstract))

overall = pd.DataFrame(
    [
        {
            "specification": "Primary: WLS + model FE + comparison-set FE",
            "self_affiliation_effect": coef,
            "std_error": se,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "p_value": p,
            "n_terms": len(x),
            "n_affiliated_terms": int(x["affiliated"].sum()),
        },
        {
            "specification": "Robustness: OLS + model FE + comparison-set FE",
            "self_affiliation_effect": uw_coef,
            "std_error": uw_se,
            "ci_low": uw_ci_low,
            "ci_high": uw_ci_high,
            "p_value": uw_p,
            "n_terms": len(x),
            "n_affiliated_terms": int(x["affiliated"].sum()),
        },
        {
            "specification": "Raw mean difference",
            "self_affiliation_effect": raw_diff,
            "std_error": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "p_value": np.nan,
            "n_terms": len(x),
            "n_affiliated_terms": int(x["affiliated"].sum()),
        },
    ]
)

display(overall)

by_model = (
    x.groupby(["model", "affiliated"])
    .agg(
        n=("estimate", "size"),
        mean_effect=("estimate", "mean"),
    )
    .reset_index()
    .pivot(index="model", columns="affiliated", values=["n", "mean_effect"])
)

by_model.columns = [
    "n_unaffiliated",
    "n_affiliated",
    "mean_unaffiliated",
    "mean_affiliated",
]

by_model["affiliated_minus_unaffiliated"] = (
    by_model["mean_affiliated"] - by_model["mean_unaffiliated"]
)

by_model = by_model.sort_values(
    "affiliated_minus_unaffiliated",
    ascending=False,
)

display(Markdown("## Per-model self-affiliation effects"))
display(by_model)


## Main conclusion

The estimated overall commercial self-affiliation effect is **+0.0144 stance-score units**
(SE = 0.0070, 95% CI [+0.0006, +0.0282], p = 0.0414).

In the primary precision-weighted model with **model** and **comparison-set** fixed effects,
affiliated model–entity pairs are therefore **more favorable toward affiliated entities** than unaffiliated pairs.
This effect is **statistically significant** at the 5% level.

For context, the raw affiliated-minus-unaffiliated difference is **+0.0130** stance-score units,
before controlling for model and comparison-set composition.

Analysis used **810** model/entity/comparison-set effects:
**65** affiliated and **745** unaffiliated.


,specification,self_affiliation_effect,std_error,ci_low,ci_high,p_value,n_terms,n_affiliated_terms
0,Primary: WLS + model FE + comparison-set FE,0.014363,0.007043,0.000558,0.028168,0.041426,810,65
1,Robustness: OLS + model FE + comparison-set FE,0.013400,0.007005,-0.000330,0.027131,0.055770,810,65
2,Raw mean difference,0.013021,NaN,NaN,NaN,NaN,810,65


## Per-model self-affiliation effects

,n_unaffiliated,n_affiliated,mean_unaffiliated,mean_affiliated,affiliated_minus_unaffiliated
model,,,,,
grok-4.1-fast,42.0,3.0,-0.003656,0.051183,0.054839
deepseek-v3.2,44.0,1.0,-0.001186,0.052167,0.053353
qwen3.5-flash-02-23,41.0,4.0,-0.004463,0.045747,0.050210
gpt-oss-120b,42.0,3.0,-0.003347,0.046856,0.050203
nemotron-3-super-120b-a12b,43.0,2.0,-0.002189,0.047060,0.049248
mistral-small-2603,41.0,4.0,-0.003170,0.032488,0.035657
grok-4,42.0,3.0,-0.002344,0.032820,0.035164
mistral-nemo,41.0,4.0,-0.002784,0.028534,0.031318
phi-4,38.0,7.0,-0.003196,0.017349,0.020545
